# Análise de dados TCP-CII

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./T CELL/DENV 2 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,LNDTWKIEKASFIEVK,206,221,16,HLA-DRB1*01:01,246,0.02,WKIEKASFI,0.982841,0.02
1,1,LNDTWKIEKASFIEVKN,206,222,17,HLA-DRB1*01:01,314,0.06,WKIEKASFI,0.971760,0.06
2,1,LNDTWKIEKASFIEV,206,220,15,HLA-DRB1*01:01,178,0.08,WKIEKASFI,0.970218,0.08
3,1,DSGCVVSWKNKELK,1,14,14,HLA-DRB1*15:01,69,0.11,VVSWKNKEL,0.892845,0.11
4,1,LNDTWKIEKASFIEVKNC,206,223,18,HLA-DRB1*01:01,382,0.12,WKIEKASFI,0.942143,0.12
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,TTASGKLITEWCCRSCTLP,301,319,19,HLA-DRB3*02:02,468,100.00,LITEWCCRS,0.000010,100.00
16412,1,TTASGKLITEWCCRSCTLPP,301,320,20,HLA-DRB3*02:02,535,100.00,LITEWCCRS,0.000007,100.00
16413,1,KLITEWCCRSCTL,306,318,13,HLA-DRB3*02:02,62,100.00,LITEWCCRS,0.000007,100.00
16414,1,TTASGKLITEWCC,301,313,13,HLA-DRB3*02:02,61,100.00,TASGKLITE,0.000006,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [3]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,LNDTWKIEKASFIEVK,206,221,16,HLA-DRB1*01:01,246,0.02,WKIEKASFI,0.982841,0.02
1,1,LNDTWKIEKASFIEVKN,206,222,17,HLA-DRB1*01:01,314,0.06,WKIEKASFI,0.971760,0.06
2,1,LNDTWKIEKASFIEV,206,220,15,HLA-DRB1*01:01,178,0.08,WKIEKASFI,0.970218,0.08
3,1,DSGCVVSWKNKELK,1,14,14,HLA-DRB1*15:01,69,0.11,VVSWKNKEL,0.892845,0.11
4,1,LNDTWKIEKASFIEVKNC,206,223,18,HLA-DRB1*01:01,382,0.12,WKIEKASFI,0.942143,0.12
...,...,...,...,...,...,...,...,...,...,...,...
581,1,WIESALNDTWKIEKASFIE,201,219,19,HLA-DRB1*07:01,448,4.90,WKIEKASFI,0.185883,4.90
582,1,AAIKDNRAVHADMGYWIES,186,204,19,HLA-DQA1*01:02/DQB1*06:02,445,4.90,NRAVHADMG,0.175116,4.90
583,1,GDIKGIMQAGKRSLRPQPTEL,91,111,21,HLA-DRB1*13:02,560,4.90,IMQAGKRSL,0.162264,4.90
584,1,QTFLIDGPETAECPNT,131,146,16,HLA-DRB3*02:02,231,4.90,FLIDGPETA,0.115049,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [4]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDNRAVHADM,186,198,2,1.110,"HLA-DRB1*13:02, HLA-DRB3*02:02"
1,AAIKDNRAVHADMG,186,199,2,1.530,"HLA-DRB1*13:02, HLA-DRB3*02:02"
2,AAIKDNRAVHADMGY,186,200,3,3.300,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB1*13:02, HLA..."
3,AAIKDNRAVHADMGYW,186,201,2,1.865,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB3*02:02"
4,AAIKDNRAVHADMGYWI,186,202,2,1.945,"HLA-DQA1*01:02/DQB1*06:02, HLA-DRB3*02:02"
...,...,...,...,...,...,...
245,YRPGYHTQITGPWHLGK,256,272,2,2.000,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
246,YRPGYHTQITGPWHLGKL,256,273,2,2.450,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
247,YRPGYHTQITGPWHLGKLE,256,274,2,2.700,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
248,YRPGYHTQITGPWHLGKLEM,256,275,1,1.800,HLA-DQA1*05:01/DQB1*02:01


In [5]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,DSGCVVSWKNKELKCG,1,16,1,0.34,HLA-DRB1*15:01
1,GVFTTNIWLKLKE,161,173,5,0.36,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
2,GVFTTNIWLKLKEK,161,174,5,0.39,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
3,DSGCVVSWKNKELKCGS,1,17,1,0.48,HLA-DRB1*15:01
4,LKYSWKTWGKAKM,111,123,1,0.50,HLA-DPA1*01:03/DPB1*02:01
...,...,...,...,...,...,...
245,EEGICGIRSVTRLE,51,64,1,4.80,HLA-DRB5*01:01
246,ITPELNHILSENEVKLT,71,87,1,4.80,HLA-DRB1*04:05
247,PKSHTLWSNGVLESEM,226,241,1,4.80,HLA-DRB3*02:02
248,DVFCDSKLMSAAIK,176,189,1,4.90,HLA-DRB1*03:01


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [6]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0           DSGCVVSWKNKELKCG
1              GVFTTNIWLKLKE
2             GVFTTNIWLKLKEK
3          DSGCVVSWKNKELKCGS
4              LKYSWKTWGKAKM
               ...          
245           EEGICGIRSVTRLE
246        ITPELNHILSENEVKLT
247         PKSHTLWSNGVLESEM
248           DVFCDSKLMSAAIK
249    GDIKGIMQAGKRSLRPQPTEL
Name: peptide, Length: 250, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [7]:
!seqkit grep -s -v -r -p '[-*X]' './Fastas/denv2_NS1_filtrado2.fasta' > DENV2_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [8]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,GVFTTNIWLKLKE,13,58.20% (504/866),84.62%,100.00%,NaN
1,2,NP 2,GVFTTNIWLKLKEK,14,23.09% (200/866),78.57%,100.00%,NaN
2,3,NP 3,GSGIFITDNVHTWTEQY,17,97.00% (840/866),94.12%,100.00%,NaN
3,4,NP 4,KRSLRPQPTELKY,13,91.69% (794/866),69.23%,100.00%,NaN
4,5,NP 5,KELKCGSGIFITDNVHTWTEQ,21,96.54% (836/866),95.24%,100.00%,NaN
...,...,...,...,...,...,...,...,...
139,140,NP 140,TNRAWNSLEVEDYGFGVFT,19,87.88% (761/866),89.47%,100.00%,NaN
140,141,NP 141,ITPELNHILSENEV,14,51.04% (442/866),85.71%,100.00%,NaN
141,142,NP 142,DVFCDSKLMSAAIKDNRAV,19,50.00% (433/866),78.95%,100.00%,NaN
142,143,NP 143,TRLENLMWKQITP,13,96.88% (839/866),69.23%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [9]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos
0,NP 1,GVFTTNIWLKLKE,13,58.20% (504/866),84.62%,100.00%,5,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
1,NP 2,GVFTTNIWLKLKEK,14,23.09% (200/866),78.57%,100.00%,5,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
2,NP 3,GSGIFITDNVHTWTEQY,17,97.00% (840/866),94.12%,100.00%,3,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
3,NP 4,KRSLRPQPTELKY,13,91.69% (794/866),69.23%,100.00%,2,"HLA-DRB4*01:01, HLA-DRB5*01:01"
4,NP 5,KELKCGSGIFITDNVHTWTEQ,21,96.54% (836/866),95.24%,100.00%,2,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01"
...,...,...,...,...,...,...,...,...
139,NP 140,TNRAWNSLEVEDYGFGVFT,19,87.88% (761/866),89.47%,100.00%,3,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*03:01/DQB1..."
140,NP 141,ITPELNHILSENEV,14,51.04% (442/866),85.71%,100.00%,2,"HLA-DQA1*03:01/DQB1*03:02, HLA-DRB1*04:05"
141,NP 142,DVFCDSKLMSAAIKDNRAV,19,50.00% (433/866),78.95%,100.00%,2,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."
142,NP 143,TRLENLMWKQITP,13,96.88% (839/866),69.23%,100.00%,2,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [10]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 1,GVFTTNIWLKLKE,13,58.20% (504/866),84.62%,100.00%,5,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",58.20
1,NP 2,GVFTTNIWLKLKEK,14,23.09% (200/866),78.57%,100.00%,5,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",23.09
2,NP 3,GSGIFITDNVHTWTEQY,17,97.00% (840/866),94.12%,100.00%,3,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",97.00
3,NP 4,KRSLRPQPTELKY,13,91.69% (794/866),69.23%,100.00%,2,"HLA-DRB4*01:01, HLA-DRB5*01:01",91.69
4,NP 5,KELKCGSGIFITDNVHTWTEQ,21,96.54% (836/866),95.24%,100.00%,2,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01",96.54
...,...,...,...,...,...,...,...,...,...
139,NP 140,TNRAWNSLEVEDYGFGVFT,19,87.88% (761/866),89.47%,100.00%,3,"HLA-DQA1*01:01/DQB1*05:01, HLA-DQA1*03:01/DQB1...",87.88
140,NP 141,ITPELNHILSENEV,14,51.04% (442/866),85.71%,100.00%,2,"HLA-DQA1*03:01/DQB1*03:02, HLA-DRB1*04:05",51.04
141,NP 142,DVFCDSKLMSAAIKDNRAV,19,50.00% (433/866),78.95%,100.00%,2,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1...",50.00
142,NP 143,TRLENLMWKQITP,13,96.88% (839/866),69.23%,100.00%,2,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1...",96.88


### Sort e filtragem por percent_match e presença em alelos

In [11]:
# Parametros
percent_match_minimo = 95.0
filtarar_por_qte_de_alelos = 5

In [12]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= percent_match_minimo]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

# Filtro do número de alelos
qte_de_alelos_result_filtered = conservacy_result_filtered[
    conservacy_result_filtered["qte_de_alelos"] >= filtarar_por_qte_de_alelos
].reset_index(drop=True)

qte_de_alelos_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 64,QYKFQPESPSKLAS,14,99.19% (859/866),92.86%,100.00%,7,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",99.19
1,NP 97,QYKFQPESPSKLA,13,99.19% (859/866),92.31%,100.00%,9,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",99.19
2,NP 47,QYKFQPESPSKLASA,15,99.19% (859/866),93.33%,100.00%,7,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",99.19
3,NP 107,HTWTEQYKFQPESPSKLASA,20,99.19% (859/866),95.00%,100.00%,6,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01, HLA...",99.19
4,NP 137,HTWTEQYKFQPESPSKLAS,19,99.19% (859/866),94.74%,100.00%,6,"HLA-DRB1*01:01, HLA-DRB1*04:01, HLA-DRB1*04:05...",99.19
5,NP 85,QYKFQPESPSKLASAI,16,98.96% (857/866),93.75%,100.00%,5,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01, HLA...",98.96
6,NP 125,HTWTEQYKFQPESPSKLASAI,21,98.96% (857/866),95.24%,100.00%,6,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01, HLA...",98.96
7,NP 119,ADMGYWIESALNDTWK,16,97.23% (842/866),68.75%,100.00%,7,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1...",97.23
8,NP 126,ADMGYWIESALNDT,14,97.23% (842/866),71.43%,100.00%,5,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1...",97.23
9,NP 98,GSGIFITDNVHTWTE,15,97.00% (840/866),93.33%,100.00%,7,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",97.00
